**Exploration notebook, not part of the pipeline.** Kept to show how the design decisions were reached.

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("transit.bronze.rt_vehicle_positions")
print(f"{bronze.count():,} bronze rows")
bronze.printSchema()

In [0]:
def flatten_vehicle_positions(df):
    """bronze.rt_vehicle_positions -> one flat row per (source_file, entity). No I/O."""
    v = "entity.vehicle"
    return df.select(
        "_source_file", "_snapshot_ts", "_dt",
        F.col("entity.id").alias("entity_id"),
        F.col(f"{v}.vehicle.id").alias("vehicle_id"),
        F.col(f"{v}.vehicle.label").alias("vehicle_label"),
        F.col(f"{v}.timestamp").alias("vehicle_ts"),
        F.col(f"{v}.current_status").alias("current_status"),
        F.col(f"{v}.current_stop_sequence").alias("stop_sequence"),
        F.col(f"{v}.stop_id").alias("stop_id"),
        F.col(f"{v}.occupancy_status").alias("occupancy_status"),
        F.col(f"{v}.occupancy_percentage").alias("occupancy_pct"),
        F.col(f"{v}.position.latitude").alias("latitude"),
        F.col(f"{v}.position.longitude").alias("longitude"),
        F.col(f"{v}.position.bearing").alias("bearing"),
        F.col(f"{v}.position.speed").alias("speed"),
        F.col(f"{v}.trip.trip_id").alias("trip_id"),
        F.col(f"{v}.trip.route_id").alias("route_id"),
        F.col(f"{v}.trip.direction_id").alias("direction_id"),
        F.col(f"{v}.trip.start_date").alias("trip_start_date"),
        F.col(f"{v}.trip.start_time").alias("trip_start_time"),
        F.col(f"{v}.trip.schedule_relationship").alias("schedule_relationship"),
        F.col(f"{v}.trip.revenue").alias("is_revenue"),
        F.col(f"{v}.trip.last_trip").alias("is_last_trip"),
    )

flat = flatten_vehicle_positions(bronze)
print(f"{flat.count():,} rows · {len(flat.columns)} columns")
flat.limit(10).display()

In [0]:
cols = [c for c in flat.columns if not c.startswith("_")]
total = flat.count()

nulls = flat.select([
    F.round(100 * F.sum(F.col(c).isNull().cast("int")) / F.lit(total), 1).alias(c)
    for c in cols
])
nulls.display()

In [0]:
print(f"rows:                       {flat.count():,}")
print(f"distinct (vehicle, ts):     {flat.select('vehicle_id','vehicle_ts').distinct().count():,}")
print(f"distinct vehicles:          {flat.select('vehicle_id').distinct().count():,}")

In [0]:
(flat.groupBy("vehicle_id", "vehicle_ts").count()
   .filter(F.col("count") > 1)
   .orderBy(F.desc("count"))
   .limit(20).display())

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("vehicle_id", "vehicle_ts").orderBy(F.desc("_snapshot_ts"))

deduped = (flat
    .filter(F.col("vehicle_id").isNotNull())
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

print(f"{flat.count():,} → {deduped.count():,}")

In [0]:
carriages = (bronze
    .select(
        "_source_file", "_snapshot_ts", "_dt",
        F.col("entity.vehicle.vehicle.id").alias("vehicle_id"),
        F.col("entity.vehicle.timestamp").alias("vehicle_ts"),
        F.col("entity.vehicle.multi_carriage_details").alias("carriages"))
    .filter(F.col("carriages").isNotNull())
    .select("_source_file", "_snapshot_ts", "_dt", "vehicle_id", "vehicle_ts",
            F.posexplode("carriages").alias("carriage_pos", "c"))
    .select("_source_file", "_snapshot_ts", "_dt", "vehicle_id", "vehicle_ts",
            "carriage_pos",
            F.col("c.label").alias("carriage_label"),
            F.col("c.carriage_sequence").alias("carriage_sequence"),
            F.col("c.occupancy_status").alias("carriage_occupancy_status"),
            F.col("c.occupancy_percentage").alias("carriage_occupancy_pct"),
            F.col("c.orientation").alias("carriage_orientation")))

print(f"{carriages.count():,} carriage rows")
carriages.limit(20).display()

In [0]:
# does the feed's own sequence agree with the array position?
(carriages.select("carriage_pos", "carriage_sequence")
   .groupBy("carriage_pos", "carriage_sequence").count()
   .orderBy("carriage_pos").display())

# which vehicles report carriages at all?
(carriages.select("vehicle_id").distinct().count())

In [0]:
(carriages.select(
    F.count("*").alias("total"),
    F.sum(F.col("carriage_orientation").isNull().cast("int")).alias("null_orientation"))
 .display())

In [0]:
(flat.filter(F.col("direction_id").isNull())
   .groupBy("route_id", "schedule_relationship", "is_revenue")
   .count().orderBy(F.desc("count")).limit(20).display())

In [0]:
(flat.groupBy("vehicle_id", "vehicle_ts").count()
   .filter(F.col("count") > 1)
   .withColumn("line", F.substring("vehicle_id", 1, 1))
   .groupBy("line")
   .agg(F.count("*").alias("dupe_groups"),
        F.round(F.avg("count"), 1).alias("avg_repeats"),
        F.max("count").alias("max_repeats"))
   .orderBy(F.desc("dupe_groups")).display())